# Training Loop for Grokking Curve Reproduction

Full training pipeline that reproduces the grokking phenomenon on (a+b) mod 97. Trains for 20,000 epochs, tracks train/test accuracy and L2 norm, and visualizes the grokking curve on log scale.

## Imports

Import visualization, tensor operations, and custom modules.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.optim import AdamW
import numpy as np

from data.modular_arithmetic import get_dataloaders
from models.transformer import Transformer

## Compute L2 Norm Function

Flatten all model parameters and compute their Euclidean norm.

In [ ]:
def compute_l2_norm(model):
    all_params = torch.cat([p.flatten() for p in model.parameters()])
    l2_norm = torch.norm(all_params).item()
    return l2_norm

## Device Setup

Detect MPS (Apple Silicon) availability; fall back to CPU if not available.

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

## Data and Model Initialization

Create data loaders with full-batch training (2822 samples per batch). Instantiate model and move to device.

In [ ]:
data_loader = get_dataloaders(97, batch_size=int(0.3 * 97 * 97))
model = Transformer(vocab_size=98, d_model=128).to(device)
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1.0)

print("Optimizer:", optimizer)
cross_entropy_loss = nn.CrossEntropyLoss()

## Initialize Tracking Lists

Set up epoch count and history lists for train/test accuracy, loss, and L2 norm.

In [ ]:
num_epochs = 20000
train_acc_history = []
test_acc_history = []
loss_history = []
l2_norm_history = []

## Main Training Loop

For each epoch: train on full batch (forward → loss → backward → step). Track train accuracy. Then evaluate on test set (forward only, no updates). Append histories and print every 100 epochs.

In [ ]:
for epoch in range(num_epochs):
    total_correct = 0
    total_samples = 0
    for x, y in data_loader[0]:
        x, y = x.to(device), y.to(device)
        logit = model.forward(x)
        equal_sign_logit = logit[:, 2, :]

        loss = cross_entropy_loss(equal_sign_logit, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        predicted = equal_sign_logit.argmax(dim=1)
        total_correct += (predicted == y).sum().item()
        total_samples += len(y)

## Test Evaluation

After each epoch, evaluate on test set without gradient updates to measure generalization.

In [ ]:
    test_total_correct = 0
    test_total_samples = 0

    for x_test, y_test in data_loader[1]:
        x_test, y_test = x_test.to(device), y_test.to(device)
        logit_test = model.forward(x_test)
        equal_sign_logit_test = logit_test[:, 2, :]
        predicted_test = equal_sign_logit_test.argmax(dim=1)
        test_total_correct += (predicted_test == y_test).sum().item()
        test_total_samples += len(y_test)

## History Tracking and Logging

Append metrics to history lists. Print progress every 100 epochs.

In [ ]:
    train_acc_history.append(total_correct / total_samples)
    test_acc_history.append(test_total_correct / test_total_samples)
    loss_history.append(loss.item())
    l2_norm_history.append(compute_l2_norm(model))
    if (epoch + 1) % 100 == 0:
        compute_l2 = l2_norm_history[-1]
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}, Train Accuracy: {train_acc_history[-1]:.4f}, Test Accuracy: {test_acc_history[-1]:.4f}, L2 Norm: {compute_l2:.4f}")

## Plot Grokking Curve

Plot train vs. test accuracy on log-scale x-axis. Save as grokking_curve.png.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), train_acc_history, label="Train Accuracy")
plt.plot(range(1, num_epochs + 1), test_acc_history, label="Test Accuracy")
plt.xscale("log")
plt.xlabel("Epoch (log scale)")
plt.ylabel("Accuracy")
plt.title("Grokking Curve")
plt.legend()
plt.savefig("grokking_curve.png")
plt.show()